In [1]:
from neo4j import GraphDatabase

URI = "neo4j://localhost"
AUTH = ("neo4j", "thomas99")

driver = GraphDatabase.driver(URI, auth=AUTH)

In [10]:
driver.execute_query("CREATE CONSTRAINT person_id IF NOT EXISTS FOR (p:Person) REQUIRE p.id IS UNIQUE")
driver.execute_query("CREATE CONSTRAINT document_id IF NOT EXISTS FOR (d:Document) REQUIRE d.id IS UNIQUE")
driver.execute_query("CREATE CONSTRAINT organization_id IF NOT EXISTS FOR (o:Organization) REQUIRE o.id IS UNIQUE")
driver.execute_query("CREATE CONSTRAINT topic_name IF NOT EXISTS FOR (t:Topic) REQUIRE t.name IS UNIQUE")

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x114705250>, keys=[])

## Populate 'Person' nodes

In [19]:
from typing import Any
from pymongo import AsyncMongoClient

from ddd_policy_tracer.discovery.hansard.talker_repository import MongoTalkerRepository


client = AsyncMongoClient[Any]("mongodb://localhost:27017")

talker_repository = MongoTalkerRepository(mongo_client=client, db_name="tmp")
talkers = await talker_repository.list_talkers()

with driver.session() as session:
    for talker in talkers:
        if talker.name == "" or talker.name is None:
            continue
        name_parts = talker.name.split(", ")
        full_name = f"{name_parts[1]} {name_parts[0]}"
        session.run(
            "CREATE (p:Person {id: $id, name: $name, electorate: $electorate, party: $party})",
            id=talker.id,
            name=full_name,
            electorate=talker.electorate,
            party=talker.party,
        )

## Populate 'Organization' nodes

Simple for now, it's only 'Australia Institute'

In [20]:
with driver.session() as session:
    session.run(
            "CREATE (o:Organization {id: $id, name: $name})",
            id='australia_institute',
            name="The Australia Institute",
        )

## Populate 'Topic' nodes

In [21]:
from ddd_policy_tracer.analysis.enrichment.topics import topics_sm

for topic, description in topics_sm.items():
    with driver.session() as session:
        session.run(
            "CREATE (t:Topic {name: $name, description: $description})",
            name=topic,
            description=description,
        )

## Populate 'Document' nodes

A Document node has two normal properties: id & text.

We also add the chunks embedding as an attribute.

In [6]:
from ddd_policy_tracer.analysis.chunks.adapters import SQLiteDocumentChunkRepository
from ddd_policy_tracer.utils.db_manager import DbManager

chunk_repository = SQLiteDocumentChunkRepository(db_manager=DbManager.get_instance("data/new2.db"))
chunks = await chunk_repository.list_chunks(source_id="committees")
filtered_chunks = [chunk for chunk in chunks if chunk.topics is not None and chunk.embedding is not None]

In [11]:
with driver.session() as session:
    for chunk in filtered_chunks:
        q = """MERGE (d:Document {id: $id, text: $text, topics: $topic_names, source_id: $source, type: $doc_type})
                UNWIND $topic_names AS topic_name
                MATCH (t:Topic {name: topic_name})
                MERGE (d)-[:DISCUSSES]->(t)
            """
        if chunk.source_id == "australia_institute":
            source = chunk.source_id
            q += """
            MATCH (o:Organization {id: $source})
            MERGE (o)-[:SOURCE_OF]->(d)
            """
        elif chunk.source_id == "hansard":
            source = chunk.source_document_id.split("_")[0]
            q += """
            MATCH (p:Person {id: $source})
            MERGE (p)-[:SOURCE_OF]->(d)
            """
        else:
            print("Unknown source, skipping relationships for chunk:", chunk.chunk_id)
            source = chunk.source_id
        # set embedding
        q += """
            CALL db.create.setNodeVectorProperty(d, 'embedding', $embedding)
        """
        session.run(
            q,
            id=chunk.chunk_id,
            text=chunk.chunk_text,
            embedding=chunk.embedding,
            topic_names=[c['topic'] for c in chunk.topics],
            source=source,
            doc_type="submission" if chunk.source_document_id != "report_recommendations" else "recommendation",
        )

Unknown source, skipping relationships for chunk: chunk_d2e41d9c1143b169
Unknown source, skipping relationships for chunk: chunk_c1c516372b19b7d5
Unknown source, skipping relationships for chunk: chunk_ca31656cfbd4803e
Unknown source, skipping relationships for chunk: chunk_ceccfbf6e7d5adbf
Unknown source, skipping relationships for chunk: chunk_07579e0b072d7061
Unknown source, skipping relationships for chunk: chunk_cfec2c151613bbee
Unknown source, skipping relationships for chunk: chunk_58d27b7c259754a6
Unknown source, skipping relationships for chunk: chunk_fbfbd0602923414b
Unknown source, skipping relationships for chunk: chunk_cfb558fcaf546cf9
Unknown source, skipping relationships for chunk: chunk_fd29a7c2684e9482
Unknown source, skipping relationships for chunk: chunk_350817773a75315e
Unknown source, skipping relationships for chunk: chunk_ca96754994b99748
Unknown source, skipping relationships for chunk: chunk_6f0340658b821be2
Unknown source, skipping relationships for chunk: c

In [16]:
filtered_chunks[0].chunk_id

'chunk_68bac771a6c84a4f'

In [ ]:
"""
MATCH (qPerson:Person {name: "Barnaby Joyce"})-[:SOURCE_OF]->(qDoc:Document)-[:DISCUSSES]->(qTopic:Topic {name: "Energy Policy"})
MATCH (d:Document)
  SEARCH d IN (
    VECTOR INDEX documentSimilarity
    FOR qDoc.embedding
    LIMIT 5
  ) SCORE AS similarity
MATCH (s)-[:SOURCE_OF]->(d)
WHERE s.name <> "Barnaby Joyce"
RETURN distinct labels(s), s.name, s.party, count(d) as doc_count
order by doc_count desc"""

# to find stance, these are the documents that person p discusses topic t. We can then look at the text of those documents to determine stance.
"""MATCH (p:Person {name: "Lidia Thorpe"})-[:SOURCE_OF]->(qDoc:Document)-[:DISCUSSES]->(qTopic:Topic {name: "Human Rights and Civil Liberties"})
RETURN qDoc"""